In [ ]:
import pandas as pd
import torch
from main import build_Q_matrix
from torch_engine import simulate_step_gpu

messi_data_transitions_df = pd.read_csv("./datasets/messi_data_Q_matrix")
Q_matrix = build_Q_matrix(messi_data_transitions_df)

# 1. Wake up the CUDA cores
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Engine running on: {device}")

# 2. Map strings to integer indices
state_names = list(Q_matrix.index)
state_to_idx = {state: i for i, state in enumerate(state_names)}
idx_to_state = {i: state for i, state in enumerate(state_names)}

# 3. Convert the Pandas matrix to a PyTorch Tensor and push it to the GPU
# We use float32 because it is the native, highly optimized format for NVIDIA GPUs
Q_tensor = torch.tensor(Q_matrix.values, dtype=torch.float32, device=device)

print(f"Q-Tensor Shape: {Q_tensor.shape}")

Q_tensor = simulate_step_gpu(current_states_idx=Q_tensor, Q=Q_matrix)